# Analyse exploratoire

Dans le notebook `manipuler_texte`, nous avons abordé quelques fonctions utiles pour travailler avec du texte. En particulier, nous avons introduit les fonctions de recherche du module `re`.

Le problème de cette méthode est qu'il faut connaître a priori les termes que l'on souhaite rechercher. Or, quand on travaille sur des entretiens ou des corpus de texte, on souhaite aussi faire émerger des termes, des thématiques, voir ce qui n'était pas prévu. Nous allons donc maintenant aborder quelques outils permettant d'explorer les entretiens.



In [ ]:
import re # Module pour utiliser les "regex", i.e. les expressions régulières
import pandas as pd
import numpy as np

## Chargement des données

Nous travaillons à partir du fichier ".txt", que nous allons ouvrir en mode "readlines()"

In [ ]:
with open("../data/gb/transcription_originale/grazia_borrini_07-06-18.txt","r", encoding='utf-8-sig') as f:
    #print(f)
    lines = f.readlines()
lines = [l for l in lines if len(l) > 1] 

# Identifier les termes "spécifiques" d'un entretien

La première méthode d'exploration que nous allons appliquer est l'identification des termes spécifiques à l'aide du "Term Frequency-Inverse Document Frequency" (TF-IDF)

## Le Term Frequency-Inverse Document Frequency (TF-IDF)

Le "Term Frequency-Inverse Document Frequency" (TF-IDF) est un indice statistique utilisé dans la recherche d'information et l'exploration de documents. Il permet de "mesurer" l'importance d'un terme dans un document comparativement à un ensemble de documents. Au sens du "TF-IDF", un mot est important dans un document lorsqu'il apparait plus fréquement que dans le reste du corpus. Par exemple, si dans un entretien l'expression "savoirs locaux" apparaît 10 fois alors que dans les autres il n'apparaît jamais, on peut supposer que c'est une expression "importante" pour la personne qui la cite.

Les éléments qui vont suivre permettent de calculer ce TF-iDF. Nous allons d'abord voir comment le calculer 'from scratch', afin de comprendre les différents élements qui entre dans le calcul de l'indice au cas où voudriez le faire dans excel par exemple. Puis, nous utiliserons des modules pythons qui permettent de le faire de façon plus "concise".

Le calcul du TF-IDF suppose au préalable de "nettoyer" les textes : le passer en petite casse, retirer les signes de ponctuation et les "stop words" (mots vides en français). Les "stop words" sont des mots tellement communs qu'ils n'apportent théoriquement aucune information sur le contenu du texte. Les articles (la, le, les, de, du, des, etc.), les auxilliaires (être, avoir) sont généralement considérés comme des stop words. 

Plusieurs modules sous Python proposent des listes de "stop words". Je vous propose d'utiliser les modules "Spacy" et "Nltk" conçues spécialement pour le traitement automatique des langues.

### Installer et charger Spacy

Spacy n'est pas encore installé. Pour cela nous ouvrons une cellule de code et écrivons `!pip install spacy`

In [ ]:
!pip install spacy

Une fois spacy installé, nous allons télécharger le modèle de langue dont on n'a besoin. Dans le cas des entretiens utilisés pour l'atelier, on prend le modèle français. Mais il existe d'autres langues : l'anglais bien sûr, mais aussi l'allemand, le chinois, le polonais, etc. 

Vous trouverez les modèles de langue existant ici  : [https://spacy.io/models](https://spacy.io/models)



In [ ]:
!python -m spacy download fr_core_news_md

### Les stop words proposés par Spacy

Enfin, on peut importer le module spacy, charger le modèle et imprimer la liste des stop words

In [ ]:
import spacy
nlp = spacy.load('fr_core_news_md')
print(f"Il y a {len(nlp.Defaults.stop_words)} termes dans la liste des stopwords de Spacy")
print(nlp.Defaults.stop_words)

### Les stop words proposés par le module nltk

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

In [ ]:
stoplist = set(stopwords.words("french"))
print(f"Il y a {len(stoplist)} termes dans la liste des stopwords de NLTK")


In [ ]:
search_speaker = re.compile(r"Speaker\s\d:")
speaker = search_speaker.match(lines[0])
search_speaker.sub("", lines[0])

La fonction `compute_tfidf` décompose l'algorithme. Pour chaque bloc, on compte le nombre de fois qu'apparaît le terme, le nombre de blocs dans lequel il est présent et le nombre de bloc total.
On calcule par ailleurs deux tf-idf : l'un avec la fréquence absolue, l'autre avec la fréquence relative.


## TF-IDF avec scikit learn

Scikit-learn est un module dédié "machine learning" et dispose d'une fonction pour calculer "facilemen" le tf-idf

In [ ]:
# importer le vectoriseur TfidfVectorizer de Scikit-Learn.  
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:

# Récupération des stopwords par défaut de SpaCy
spacy_stopwords = set(nlp.Defaults.stop_words)




La fonction ci-dessous permet de lemmatiser et retirer les stopwords. On peu choisir si on préfère travailler avec les stopwords de NLTK ou Spacy.

In [ ]:
# Fonction pour lemmatiser le texte et retirer les stopwords
def lemmatize_and_remove_stopwords(text, stoplist, lemmatiser = False):
    doc = nlp(text)
    if lemmatiser == True:
        cleaned_text = " ".join([token.lemma_ for token in doc if token.text.lower() not in stoplist and not token.is_punct])
    else:
        cleaned_text = " ".join([str(token) for token in doc if token.text.lower() not in stoplist and not token.is_punct])
    return cleaned_text

In [ ]:
lemmatize_and_remove_stopwords(lines[0], nlp, stoplist = stoplist, lemmatiser = True)

In [ ]:
lines[0]

In [ ]:
tous_documents = []
for line in lines:
    lemmatized_an_cleaned_text = lemmatize_and_remove_stopwords(line, nlp, stoplist = stoplist, lemmatiser = False)
    tous_documents.append(lemmatized_an_cleaned_text)


vectoriseur = TfidfVectorizer(use_idf=True)
documents_transformes = vectoriseur.fit_transform(tous_documents)

In [ ]:
documents_transformes_tableau = documents_transformes.toarray()
len(documents_transformes_tableau)

In [ ]:
i = -1
for compteur, document in enumerate(documents_transformes_tableau):
    i+=1
    #print("#####\n",document)
    # construire un objet de la classe DataFrame
    tf_idf_tuples = list(zip(vectoriseur.get_feature_names_out(), document))
    df0 = pd.DataFrame.from_records(tf_idf_tuples, columns=['terme', 'tfidf']).sort_values(by='tfidf', ascending=False).reset_index(drop=True)
    df0 = df0.loc[df0.tfidf > 0]
    df0['bloc_id'] = str(compteur)
    if i == 0:
        df = df0.copy()
    else:
        df = pd.concat([df, df0])

for bloc in df.bloc_id.unique():
    df_bloc = df.loc[df.bloc_id == bloc].sort_values("tfidf", ascending=False)
    specific_term = [x for x in df_bloc.terme.head(5)]
    print(f"Les 5 termes les plus spécifique du bloc {bloc} sont : ", specific_term)


In [ ]:
df

## Comparaison des deux entretiens

In [ ]:
gb["itw"] = "itw_gb"
mr["itw"] = "itw_mr"


text_gb = [x for x in gb.text]
text_mr = [x for x in mr.text]

In [ ]:
tous_documents = []
for itw in [text_gb, text_mr]:
    lemmatized_an_cleaned_text = lemmatize_and_remove_stopwords(" ".join(itw), nlp, spacy_stopwords, lemmatiser = True)
    tous_documents.append(lemmatized_an_cleaned_text)

vectoriseur = TfidfVectorizer(max_df=.65, min_df=1, use_idf=True)
documents_transformes = vectoriseur.fit_transform(tous_documents)

In [ ]:
documents_transformes_tableau = documents_transformes.toarray()
len(documents_transformes_tableau)

In [ ]:
i = -1
for compteur, document in enumerate(documents_transformes_tableau):
    i+=1
    #print("#####\n",document)
    # construire un objet de la classe DataFrame
    tf_idf_tuples = list(zip(vectoriseur.get_feature_names_out(), document))
    df0 = pd.DataFrame.from_records(tf_idf_tuples, columns=['terme', 'tfidf']).sort_values(by='tfidf', ascending=False).reset_index(drop=True)
    df0 = df0.loc[df0.tfidf >= 0]
    df0['bloc_id'] = str(compteur)
    if i == 0:
        df = df0.copy()
    else:
        df = pd.concat([df, df0])

for bloc in df.bloc_id.unique():
    df_bloc = df.loc[df.bloc_id == bloc].sort_values("tfidf", ascending=False)
    specific_term = [x for x in df_bloc.terme.tail(20)]
    print(f"Les 5 termes les plus spécifique du bloc {bloc} sont : ", specific_term)


In [ ]:
df.loc[df.terme.str.contains(r"participat")]

## Décortiquer le TF-IDF

Cette partie s'inspire de l'article "Analyse de documents avec TF-IDF" de Matthew Lavin publié dans la revue en ligne *Programming Historian* (revue que je recommande pour ces nombreux articles didactiques).

La formule mathématique pour calculer le TF-IDF est la suivante :

\begin{equation}
Tf{\text -}Idf_i = Tf_i \times Idf_i
\end{equation}

* $Tf_i$ : la fréquence du terme *i* dans le document. Selon les algorithmes, la fréquence correspond à la fréquence absolue ou à la fréquence relative (i.e. fréquence du terme *i* divisé par le nombre de tokens dans le document)
* $Idf_i$ : la fréquence *inverse* de documents (inverse document frequency) du terme *i*. Elle correspond au *logarithme naturel* du nombre de documents analysés divisé par le nombre de documents contenant le terme *i*.

$$
Idf_i = ln\left[{\frac{N}{df_i}}\right]
$$

* N : le nombre de document analysés
* df_i : le nombre de document contenant *i*.

Souvent, les algorythmes implémentés dans les modules informatiques "normalisent" l'idf en ajoutant $+1$ au numérateur puis au logarithme. C'est le cas du module Scikit-learn que nous allons utilisé plus bas.

$$
Idf_i = ln\left[{\frac{N+1}{df_i}}\right]+1
$$

Si la formule peut impressionnée, elle n'a rien de vraiment compliqué puisque la plupart des opérations sont des additions, mutliplications et divisions. En ce qui concerne le logarithme, nous allons utilisez le module numpy.


Un petit exemple concret :

On remarque que dans un entretien le mot "domination" apparaît 10 fois. Ce mot apparaît par ailleurs dans 5 entretiens et notre corpus en contient 30.
On remarque également que dans ce même entretien le mot "famille" apparaît 20 fois et qu'il est présent dans les 30 entretiens.

|terme | $tf_i$ | $df_i$ | $N$| $tf{\text -}idf_i$|
|------|--------|--------|----|-----------|
|domination| 10 | 5 | 30 | 28,25 |
|famille | 20 | 30 | 30 | 20,66 |

$$
tf{\text -}idf_{domination} = 10 \times \left[len\left(\frac{30+1}{5}\right) +1\right] \approx 28,25 
$$

$$
tf{\text -}idf_{famille} = 20 \times \left[len\left(\frac{30+1}{30}\right) +1\right] \approx 20,66
$$

